# 18 · Transformers: tokenization, embeddings, and causal attention

**Prerequisites:** 00 — Setup; basic Python. No other notebook state is required.

**Execution:** portable. Run cells from top to bottom. Each notebook is independent.

## What you will learn

Implement a single attention head; inspect masking; understand what a pretrained model adds.


## Step 1 — Represent tokens

This is a tiny randomly initialized attention demonstration. It is not a pretrained LLM and will not generate useful prose.


In [ ]:
import numpy as np
rng=np.random.default_rng(42)
tokens=['data','powers','useful','AI']
d=8
E=rng.normal(size=(len(tokens),d))
positions=np.arange(len(tokens))[:,None]
frequencies=1/10000**(np.arange(d)[None,:]/d)
H=E+np.sin(positions*frequencies)
Wq=rng.normal(size=(d,d)); Wk=rng.normal(size=(d,d)); Wv=rng.normal(size=(d,d))
print('Token embedding shape:',H.shape)


**Check your result:** Each of four tokens has an eight-dimensional representation.


## Step 2 — Compute masked attention

Scaled dot products measure query/key compatibility. The upper-triangular mask prevents a token from looking into the future.


In [ ]:
Q,K,V=H@Wq,H@Wk,H@Wv
scores=Q@K.T/np.sqrt(d)
future=np.triu(np.ones_like(scores,dtype=bool),k=1)
scores=np.where(future,-np.inf,scores)
exp=np.exp(scores-np.max(scores,axis=1,keepdims=True))
attention=exp/exp.sum(axis=1,keepdims=True)
output=attention@V
assert np.allclose(attention.sum(axis=1),1)
assert np.all(attention[future]==0)
print(np.round(attention,3))


**Check your result:** Attention rows sum to 1 and future positions have zero weight.


## Step 3 — Add residual connection and layer normalization

Production transformers stack attention heads, feed-forward blocks and normalization. Pretraining learns their parameters over large corpora.


In [ ]:
residual=H+output
normalized=(residual-residual.mean(axis=1,keepdims=True))/np.sqrt(residual.var(axis=1,keepdims=True)+1e-5)
assert normalized.shape==H.shape
print('Per-token means:',normalized.mean(axis=1))


**Check your result:** Normalized token means are close to zero.


## Practice

Remove the causal mask and compare attention. Explain encoder bidirectional attention versus decoder causal attention.

Predict the outcome before changing the code. Explain the result in your own words.


## Solution guide

Without the mask, every token can use future tokens. Autoregressive next-token training requires causal masking to prevent target leakage.

## Before using this at work

Use representative data, measure quality on held-out examples, record dependencies and permissions, and review failures. Tiny synthetic results demonstrate mechanics; they do not establish production quality.
